# Retail Orders — Data Quality Profiling

**Internship Project:** KPI Dictionary & Data Quality Contract  
**Domain:** Commerce / Retail  
**Decision Owner:** Sales / Operations Manager  

This notebook profiles the retail orders dataset for **completeness, uniqueness, validity, consistency, and freshness**. The checks are executable and produce PASS/FAIL results against defined thresholds.

In [ ]:
import pandas as pd
import numpy as np

FILE = 'retail-orders-raw (1)(2).csv'
df = pd.read_csv(FILE)

print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print('\nColumns:')
print(df.columns.tolist())

display(df.head())

## 1. Completeness Check

Required fields should not contain missing values. The project threshold is **≤ 2% missing values** in required fields.

In [ ]:
required_columns = [
    'order_id', 'order_date', 'customer_segment', 'city',
    'category', 'quantity', 'unit_price', 'payment_status'
]

missing_counts = df[required_columns].isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)

completeness = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
})
display(completeness)

critical_missing_pct = completeness['Missing %'].max()
completeness_status = 'PASS' if critical_missing_pct <= 2 else 'FAIL'
print(f'Completeness status: {completeness_status} (maximum missing = {critical_missing_pct:.2f}%)')

## 2. Uniqueness Check

`order_id` is expected to be unique. The failure threshold is **> 1% duplicate Order IDs**.

In [ ]:
duplicate_order_ids = df['order_id'].duplicated(keep=False).sum()
duplicate_order_id_pct = duplicate_order_ids / len(df) * 100
exact_duplicate_rows = df.duplicated().sum()

uniqueness_status = 'PASS' if duplicate_order_id_pct <= 1 else 'FAIL'

print(f'Duplicate Order ID records: {duplicate_order_ids}')
print(f'Duplicate Order ID rate: {duplicate_order_id_pct:.2f}%')
print(f'Exact duplicate rows: {exact_duplicate_rows}')
print(f'Uniqueness status: {uniqueness_status}')

## 3. Validity Checks

Rules: `quantity` must be a positive integer; `discount_pct` must be between 0 and 100; `order_date` must be a valid date.

In [ ]:
parsed_dates = pd.to_datetime(df['order_date'], errors='coerce')
invalid_date_count = parsed_dates.isna().sum()

quantity_num = pd.to_numeric(df['quantity'], errors='coerce')
invalid_quantity_mask = (
    quantity_num.isna() |
    (quantity_num <= 0) |
    (quantity_num % 1 != 0)
)
invalid_quantity_count = invalid_quantity_mask.sum()

discount_num = pd.to_numeric(df['discount_pct'], errors='coerce')
invalid_discount_mask = discount_num.notna() & ((discount_num < 0) | (discount_num > 100))
invalid_discount_count = invalid_discount_mask.sum()

validity_results = pd.DataFrame({
    'Check': ['Valid order date', 'Positive integer quantity', 'Discount between 0 and 100'],
    'Invalid Records': [invalid_date_count, invalid_quantity_count, invalid_discount_count]
})
validity_results['Invalid %'] = (validity_results['Invalid Records'] / len(df) * 100).round(2)
validity_results['Status'] = np.where(validity_results['Invalid %'] <= 1, 'PASS', 'FAIL')
display(validity_results)

print('Validity status:', 'PASS' if (validity_results['Status'] == 'PASS').all() else 'FAIL')

## 4. Consistency Check

Text values should use a standardized representation. This notebook normalizes whitespace/capitalization and reports the values before and after standardization.

In [ ]:
text_columns = ['customer_segment', 'payment_status', 'city', 'category']

consistency_summary = []
for col in text_columns:
    before = df[col].dropna().astype(str).unique().tolist()
    normalized = df[col].dropna().astype(str).str.strip().str.title().unique().tolist()
    consistency_summary.append({
        'Column': col,
        'Distinct Before': len(before),
        'Distinct After Standardization': len(normalized),
        'Values Before': ', '.join(map(str, before)),
        'Values After': ', '.join(map(str, normalized))
    })

consistency_df = pd.DataFrame(consistency_summary)
display(consistency_df)

print('Review the standardized values above before applying them to production data.')

## 5. Freshness Check

For this project, daily reporting data should be no more than **24 hours old** relative to the reporting reference time.

In [ ]:
# Use the latest valid order date as the dataset's freshness date.
latest_valid_date = parsed_dates.max()
print('Latest valid order date:', latest_valid_date)

# Set the reporting reference date when running the notebook.
REPORTING_DATE = pd.Timestamp.today().normalize()
age_days = (REPORTING_DATE - latest_valid_date.normalize()).days if pd.notna(latest_valid_date) else np.nan
freshness_status = 'PASS' if pd.notna(age_days) and age_days <= 1 else 'FAIL'

print('Reporting reference date:', REPORTING_DATE.date())
print('Data age (days):', age_days)
print('Freshness status:', freshness_status)

## 6. Overall Data Quality Scorecard

A critical quality failure means the dataset should be reviewed before being used for KPI reporting.

In [ ]:
scorecard = pd.DataFrame({
    'Dimension': ['Completeness', 'Uniqueness', 'Validity', 'Consistency', 'Freshness'],
    'Threshold': ['≤ 2% missing', '≤ 1% duplicate Order IDs', '≤ 1% invalid records per rule', 'Standardized categorical values', '≤ 1 day old'],
    'Status': [
        completeness_status,
        uniqueness_status,
        'PASS' if (validity_results['Status'] == 'PASS').all() else 'FAIL',
        'REVIEW',
        freshness_status
    ]
})

display(scorecard)
overall = 'PASS' if (scorecard['Status'].isin(['FAIL'])).sum() == 0 else 'FAIL'
print('Overall data quality status:', overall)

## Interpretation

The results from this notebook should be used with the **Data Quality Contract**. If a threshold fails, the responsible data owner should investigate and correct/reload the source data before publishing KPI results. The notebook is designed to be rerun whenever a new retail orders file is received.